# ISLES24 Clinical Data Explorer

Interactive Jupyter Notebook viewer for `isles24_summary.xlsx`.

Choose a comparison and variable. Continuous variables are shown as reconstructed boxplots from the reported summary statistics; categorical/ordinal variables are shown as grouped bar charts.

### Setup

Run this notebook from the ISLES24 repository. The workbook can be either:

- `isles24_summary.xlsx` in the project root, or
- `utils/isles24_summary.xlsx`.

Then use **Kernel → Restart Kernel and Run All Cells** (or **Run All Cells**) in Jupyter Notebook/JupyterLab.

> **Important:** the workbook contains aggregate summaries rather than patient-level observations. Continuous boxplots therefore use the reported Q1, median, Q3, and either 5th/95th percentiles or min/max as whiskers.


In [ ]:
# Required packages for local Jupyter.
# If any are missing, install them into the current kernel automatically.
import importlib.util
import subprocess
import sys

_required = {
    "pandas": "pandas",
    "openpyxl": "openpyxl",
    "plotly": "plotly",
    "ipywidgets": "ipywidgets",
}

_missing = [
    package
    for module, package in _required.items()
    if importlib.util.find_spec(module) is None
]

if _missing:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *_missing]
    )


In [ ]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.colors import qualitative
from IPython.display import display

try:
    import ipywidgets as widgets
except ModuleNotFoundError:
    widgets = None


In [ ]:
import os
from pathlib import Path

def find_workbook():
    env_path = os.environ.get("ISLES24_SUMMARY_XLSX")
    if env_path:
        candidate = Path(env_path).expanduser()
        if candidate.exists():
            return candidate.resolve()

    candidates = [
        Path("isles24_summary.xlsx"),
        Path("utils/isles24_summary.xlsx"),
        Path("../isles24_summary.xlsx"),
        Path("../utils/isles24_summary.xlsx"),
        Path("../../isles24_summary.xlsx"),
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    raise FileNotFoundError(
        "Could not find isles24_summary.xlsx. Put it in the ISLES24 project "
        "root or in utils/, or set ISLES24_SUMMARY_XLSX."
    )

workbook_path = find_workbook()
print(f"Using workbook: {workbook_path}")


In [ ]:
SHEETS = [
    "All Data",
    "Stratified (Train vs Test)",
    "Stratified (Center 1 vs 2)",
]

DISPLAY_NAMES = {
    "All Data": "All data",
    "Stratified (Train vs Test)": "Train vs Test",
    "Stratified (Center 1 vs 2)": "Center 1 vs Center 2",
}

xls = pd.ExcelFile(workbook_path)
print("Available sheets:", xls.sheet_names)

data = {}
for sheet in SHEETS:
    if sheet not in xls.sheet_names:
        continue

    df = pd.read_excel(workbook_path, sheet_name=sheet)
    df.columns = [str(c).strip() for c in df.columns]

    # Normalize the fourth column (Subset/Center) to a common name.
    if "Group" not in df.columns and len(df.columns) >= 4:
        df = df.rename(columns={df.columns[3]: "Group"})

    data[sheet] = df
    print(f"{sheet}: {df.shape}")

if not data:
    raise RuntimeError("None of the expected ISLES24 summary sheets were found.")

In [ ]:
def as_number(value):
    if pd.isna(value) or str(value).strip() in {"", "-", "NA", "N/A", "nan"}:
        return np.nan
    return pd.to_numeric(value, errors="coerce")


FREQ_RE = re.compile(
    r"(?:^|,\s*)(.*?):\s*(\d+)\s*\(([\d.]+)%\)(?=,\s*.*?:\s*\d+\s*\(|$)"
)


def parse_frequency(value):
    if pd.isna(value):
        return []
    return [
        {"Label": label.strip(), "Count": int(count), "Percent": float(pct)}
        for label, count, pct in FREQ_RE.findall(str(value).strip())
    ]


def unique_nonempty(series):
    result = []
    for value in series.dropna():
        text = str(value).strip()
        if text and text != "-" and text not in result:
            result.append(text)
    return result


def detect_kind(rows):
    if "Median" in rows.columns:
        vals = pd.to_numeric(rows["Median"], errors="coerce")
        if vals.notna().any():
            return "continuous"

    if "Frequency" in rows.columns and len(rows):
        lengths = rows["Frequency"].apply(parse_frequency).map(len)
        if len(lengths) and lengths.max() > 0:
            return "categorical"

    return "unknown"


def reported_p_value(rows):
    if "p-value" not in rows.columns:
        return None, None

    values = unique_nonempty(rows["p-value"])
    if not values:
        return None, None

    raw = values[0]
    match = re.search(r"([0-9]*\.?[0-9]+(?:[eE][-+]?\d+)?)", raw)
    if not match:
        return None, raw

    try:
        return float(match.group(1)), raw
    except ValueError:
        return None, raw


def significance_label(rows):
    p, raw = reported_p_value(rows)
    if p is None or p >= 0.05:
        return None

    if p < 0.001:
        stars = "***"
    elif p < 0.01:
        stars = "**"
    else:
        stars = "*"

    if str(raw).lstrip().startswith("<"):
        return f"{stars}  p {raw}"
    return f"{stars}  p = {raw}"


def reported_test_text(rows):
    parts = []

    if "Statistical Test" in rows.columns:
        tests = unique_nonempty(rows["Statistical Test"])
        if tests:
            parts.append(f"Statistical test: {tests[0]}")

    if "p-value" in rows.columns:
        pvals = unique_nonempty(rows["p-value"])
        if pvals:
            parts.append(f"p-value: {pvals[0]}")

    return " | ".join(parts)


def add_significance_bracket(fig, rows, y_top, y_span):
    label = significance_label(rows)
    if not label or len(rows) < 2:
        return fig

    pad = max(abs(y_span) * 0.07, 1e-9)
    y = y_top + pad
    h = max(abs(y_span) * 0.035, pad * 0.45)

    # Paper x-coordinates span the two compared groups cleanly.
    x0, x1 = 0.18, 0.82

    for xa, xb, ya, yb in [
        (x0, x0, y - h, y),
        (x0, x1, y, y),
        (x1, x1, y, y - h),
    ]:
        fig.add_shape(
            type="line",
            xref="paper",
            yref="y",
            x0=xa,
            x1=xb,
            y0=ya,
            y1=yb,
            line=dict(width=1.5),
        )

    fig.add_annotation(
        xref="paper",
        yref="y",
        x=0.5,
        y=y + h * 0.20,
        text=label,
        showarrow=False,
        yanchor="bottom",
        font=dict(size=13),
    )

    fig.update_yaxes(range=[None, y + h * 2.2])
    return fig

In [ ]:
GROUP_COLORS = qualitative.Safe


def plot_continuous(rows, variable, whisker_mode="5th/95th percentiles"):
    rows = rows.copy()

    rows["median_num"] = rows["Median"].apply(as_number)
    rows["q1_num"] = rows["25th Percentile (Q1)"].apply(as_number)
    rows["q3_num"] = rows["75th Percentile (Q3)"].apply(as_number)

    if whisker_mode == "Min/max":
        lower_col, upper_col = "Min", "Max"
    else:
        lower_col, upper_col = "5th Percentile", "95th Percentile"

    rows["lower_num"] = rows[lower_col].apply(as_number)
    rows["upper_num"] = rows[upper_col].apply(as_number)
    rows["lower_num"] = rows["lower_num"].fillna(rows["q1_num"])
    rows["upper_num"] = rows["upper_num"].fillna(rows["q3_num"])

    rows = rows.dropna(subset=["median_num", "q1_num", "q3_num"])

    if rows.empty:
        raise ValueError("No numeric summary statistics are available for this selection.")

    fig = go.Figure()
    groups = [str(row.get("Group", "All data")) for _, row in rows.iterrows()]
    x_positions = list(range(len(groups)))

    for i, (_, row) in enumerate(rows.iterrows()):
        color = GROUP_COLORS[i % len(GROUP_COLORS)]

        # Draw the reconstructed box without a mean line.
        fig.add_trace(
            go.Box(
                x=[i],
                name=groups[i],
                q1=[row["q1_num"]],
                median=[row["median_num"]],
                q3=[row["q3_num"]],
                lowerfence=[row["lower_num"]],
                upperfence=[row["upper_num"]],
                boxmean=False,
                boxpoints=False,
                width=0.48,
                fillcolor=color,
                line=dict(color=color, width=2),
                marker=dict(color=color),
                opacity=0.75,
                hovertemplate=(
                    f"<b>{groups[i]}</b><br>"
                    "Q1: %{q1}<br>"
                    "Median: %{median}<br>"
                    "Q3: %{q3}<br>"
                    "Lower whisker: %{lowerfence}<br>"
                    "Upper whisker: %{upperfence}<extra></extra>"
                ),
                showlegend=False,
            )
        )

        # Explicit 50th-percentile line so the median is always clearly visible.
        fig.add_shape(
            type="line",
            xref="x",
            yref="y",
            x0=i - 0.24,
            x1=i + 0.24,
            y0=row["median_num"],
            y1=row["median_num"],
            line=dict(color="black", width=2.5),
            layer="above",
        )

    y_min = float(np.nanmin(rows["lower_num"]))
    y_max = float(np.nanmax(rows["upper_num"]))
    y_span = max(y_max - y_min, abs(y_max) * 0.1, 1.0)

    fig.update_layout(
        title=str(variable),
        xaxis_title="Group",
        yaxis_title=str(variable),
        template="plotly_white",
        height=540,
        showlegend=False,
        margin=dict(l=60, r=30, t=90, b=60),
    )
    fig.update_xaxes(
        tickmode="array",
        tickvals=x_positions,
        ticktext=groups,
        range=[-0.6, len(groups) - 0.4],
    )

    if len(rows) >= 2:
        add_significance_bracket(fig, rows, y_max, y_span)

    return fig

def plot_categorical(rows, variable, percentage=True):
    fig = go.Figure()
    added = 0
    plotted_values = []

    for i, (_, row) in enumerate(rows.iterrows()):
        parsed = parse_frequency(row.get("Frequency"))
        if not parsed:
            continue

        values = [p["Percent"] if percentage else p["Count"] for p in parsed]
        plotted_values.extend(values)
        color = GROUP_COLORS[i % len(GROUP_COLORS)]

        fig.add_trace(
            go.Bar(
                name=str(row.get("Group", "All data")),
                x=[p["Label"] for p in parsed],
                y=values,
                marker_color=color,
                opacity=0.85,
            )
        )
        added += 1

    if not added:
        raise ValueError("No categorical frequency data are available for this selection.")

    y_max = max(plotted_values) if plotted_values else 1.0
    y_min = min(plotted_values) if plotted_values else 0.0
    y_span = max(y_max - y_min, y_max * 0.1, 1.0)

    fig.update_layout(
        title=str(variable),
        barmode="group",
        bargap=0.22,
        bargroupgap=0.08,
        xaxis_title=str(variable),
        yaxis_title="Percent (%)" if percentage else "Count",
        template="plotly_white",
        height=540,
        margin=dict(l=60, r=30, t=90, b=80),
        legend_title_text="Group",
    )

    if percentage:
        fig.update_yaxes(rangemode="tozero")

    if added >= 2:
        add_significance_bracket(fig, rows, y_max, y_span)

    return fig


## Interactive explorer

Run this cell once. A plot appears immediately and changing a selector redraws it automatically.

- **Comparison**: all data, train vs test, or center 1 vs center 2.
- **Variable**: the clinical variable to visualize.
- **Timepoint**: shown only if that variable occurs at more than one timepoint.
- **Whiskers**: shown only for continuous variables.
- **Display**: shown only for categorical variables.
- Significant comparisons (`p < 0.05`) are annotated with a bracket and significance stars.


In [ ]:
if widgets is None:
    print(
        "ipywidgets is not installed in this notebook kernel.\n"
        "Install it with: pip install ipywidgets\n"
        "Then restart the kernel and run all cells again."
    )
else:
    comparison = widgets.Dropdown(
        options=[(DISPLAY_NAMES[s], s) for s in SHEETS if s in data],
        description="Comparison:",
        style={"description_width": "100px"},
        layout=widgets.Layout(width="460px"),
    )

    variable = widgets.Dropdown(
        options=[],
        description="Variable:",
        style={"description_width": "100px"},
        layout=widgets.Layout(width="620px"),
    )

    timepoint = widgets.Dropdown(
        options=[],
        description="Timepoint:",
        style={"description_width": "100px"},
        layout=widgets.Layout(width="460px"),
    )

    whiskers = widgets.Dropdown(
        options=["5th/95th percentiles", "Min/max"],
        value="5th/95th percentiles",
        description="Whiskers:",
        style={"description_width": "100px"},
        layout=widgets.Layout(width="460px"),
    )

    categorical_display = widgets.ToggleButtons(
        options=[("Percent", True), ("Count", False)],
        value=True,
        description="Display:",
        style={"description_width": "100px"},
    )

    stats = widgets.HTML()
    output = widgets.Output()
    updating = False

    def current_df():
        return data[comparison.value]

    def variable_options():
        df = current_df()
        pairs = []
        table = df[["Variable", "Category"]].dropna(
            subset=["Variable"]
        ).drop_duplicates()

        for _, row in table.iterrows():
            v = str(row["Variable"]).strip()
            c = str(row["Category"]).strip() if pd.notna(row["Category"]) else ""
            label = f"{v}  [{c}]" if c and c != "-" else v
            pairs.append((label, v))

        seen = set()
        result = []
        for label, value in sorted(pairs, key=lambda x: x[0].lower()):
            if value not in seen:
                seen.add(value)
                result.append((label, value))
        return result

    def rows_for_variable():
        df = current_df()
        return df[
            df["Variable"].astype(str).str.strip()
            == str(variable.value).strip()
        ].copy()

    def selected_rows():
        rows = rows_for_variable()

        if timepoint.layout.display != "none" and timepoint.value not in (None, ""):
            rows = rows[
                rows["Timepoint"].astype(str).str.strip()
                == str(timepoint.value).strip()
            ].copy()

        return rows

    def update_visibility():
        kind = detect_kind(selected_rows())
        whiskers.layout.display = "" if kind == "continuous" else "none"
        categorical_display.layout.display = "" if kind == "categorical" else "none"

    def draw(*_):
        if updating or variable.value is None:
            return

        rows = selected_rows()
        update_visibility()

        with output:
            output.clear_output(wait=True)

            if rows.empty:
                print("No data available for this selection.")
                return

            try:
                kind = detect_kind(rows)

                if kind == "continuous":
                    fig = plot_continuous(rows, variable.value, whiskers.value)
                elif kind == "categorical":
                    fig = plot_categorical(
                        rows,
                        variable.value,
                        categorical_display.value,
                    )
                else:
                    print(
                        "This variable cannot currently be plotted from the "
                        "available summary fields."
                    )
                    stats.value = ""
                    return

                fig.show()

                test_text = reported_test_text(rows)
                stats.value = f"<b>{test_text}</b>" if test_text else ""

            except Exception as exc:
                stats.value = ""
                print(f"Could not plot this selection: {exc}")

    def refresh_timepoints(*_):
        global updating

        if variable.value is None:
            return

        updating = True
        rows = rows_for_variable()
        vals = (
            unique_nonempty(rows["Timepoint"])
            if "Timepoint" in rows.columns
            else []
        )

        timepoint.options = vals or [""]
        timepoint.value = vals[0] if vals else ""
        timepoint.layout.display = "" if len(vals) > 1 else "none"

        updating = False
        draw()

    def refresh_variables(*_):
        global updating

        updating = True
        opts = variable_options()
        variable.options = opts

        if opts:
            variable.value = opts[0][1]

        updating = False
        refresh_timepoints()

    comparison.observe(refresh_variables, names="value")
    variable.observe(refresh_timepoints, names="value")
    timepoint.observe(draw, names="value")
    whiskers.observe(draw, names="value")
    categorical_display.observe(draw, names="value")

    dashboard = widgets.VBox([
        widgets.HTML(
            "<h3 style='margin-bottom:4px'>ISLES24 Clinical Data Explorer</h3>"
            "<div style='color:#666;margin-bottom:10px'>"
            "Choose a comparison and variable. The figure updates automatically."
            "</div>"
        ),
        comparison,
        variable,
        timepoint,
        whiskers,
        categorical_display,
        stats,
        output,
    ])

    display(dashboard)
    refresh_variables()


## Notes

- The intended workflow for non-programmers is simply **Run All**, then use the interactive controls.
- Each comparison group has a distinct color.
- Continuous plots show **only the median line inside each box**; the mean is not drawn.
- Continuous plots are reconstructed from aggregate summary statistics.
- Categorical plots are parsed from the workbook's `Frequency` field.
- Reported statistical tests and p-values are displayed when available.
- Significance brackets are added automatically when the reported comparison has `p < 0.05`.